**Purpose:** `v3.2.X_reddit+news.ipynb` — part of `03_portfolio/LSTM_sharpe` (see the stage README).

**Outputs:** `LSTMvia-sharpe_reddit+news_weights.npy`, `{name}.csv`, `{prefix}_cumret.png`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.seeds import SEED_PORTFOLIO_LSTM_SHARPE_V3_2_X_REDDIT_NEWS, SEED_PORTFOLIO_LSTM_SHARPE_V3_2_X_REDDIT_NEWS_FINAL


In [ ]:
from src.config import PROJECT_ROOT


# LSTM Direct-Sharpe — Combined News + Reddit Sentiment (v3.4)

Extends v3.3 by combining **both** news (GDELT) and Reddit sentiment features
into a single model. All feature choices are fully frozen — Optuna only tunes
hyperparameters.

**Feature freezing strategy**
- Load best trial from **news study DB** → frozen technical picks + news sentiment cols + NaN strategy
- Load best trial from **reddit study DB** → frozen technical picks + reddit sentiment cols + NaN strategy
- Technical base features: union of both frozen pick sets (deduplicated)
- Sentiment features: all news cols + `_NEWS_NAN` + all reddit cols + `_REDDIT_NAN`
- Everything is fixed — Optuna searches **hyperparameters only**

**Feature layout in X_raw per asset**
```
[base_feats (union) | news_sentiment_cols | _NEWS_NAN | reddit_sentiment_cols | _REDDIT_NAN]
```

**`_NEWS_NAN` and `_REDDIT_NAN`** are single global columns (not per-asset).
Each is appended as an `identity`-normalised feature, broadcast to all N assets.

**Optuna search space** — hyperparameters only:
`lookback`, `hidden_size`, `num_layers`, `dropout`, `bidirectional`, `encoder_proj`,
`asset_mixer_layers`, `asset_mixer_heads`, `fusion_hidden`, `weight_mode`,
`learning_rate`, `weight_decay`, `batch_size`, `max_epochs`, `patience`,
`min_epochs`, `grad_clip`, `scheduler`, `sharpe_eps`, `min_delta`


## 1) Imports

In [ ]:
from __future__ import annotations

import copy
import json
import logging
import math
import os
import random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler
from optuna.trial import TrialState

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 2) Config

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
DATASET_PATH = Path(str(PROJECT_ROOT / '03_portfolio/dataset.parquet'))
SPREADS_PATH = Path(str(PROJECT_ROOT / '01_data/aux/bid-ask_spread.json'))

# Optuna DBs from the news and reddit sentiment studies
# Each DB's best trial provides frozen picks + sentiment cols + NaN strategy
NEWS_DB_PATH   = 'v322_news/sharpe_lstm_sentiment.db'
REDDIT_DB_PATH = 'v322_reddit/sharpe_lstm_reddit.db'

# ── Universe ──────────────────────────────────────────────────────────────
ASSETS = ['XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']

# ── Base feature buckets (needed to reconstruct picks from the DBs) ───────
FEATURES = {
    'Price':      ['Close'],
    'OHLC':       ['Open', 'High', 'Low'],
    'Trend':      ['SMA5', 'SMA22', 'SMA63', 'SMA126', 'SMA252', 'MACD', 'ADX'],
    'Momentum':   ['RSI', 'MACD', 'CCI'],
    'Volume':     ['Volume', 'OBV', 'ADI', 'VolumeVariation'],
    'Volatility': ['BBUpper', 'BBLower', '5dayVolatility', '22dayVolatility',
                   '63dayVolatility', 'RatioVolatility'],
    'Market':     ['VIXIndexClose'],
}

# ── Study settings ─────────────────────────────────────────────────────────
OUT_DIR     = 'v322_reddit+news'
N_TRIALS    = 0 # hyperparameter-only search; set before running
MODEL_SEEDS = [11, 22, 33]
FINAL_SEED  = SEED_PORTFOLIO_LSTM_SHARPE_V3_2_X_REDDIT_NEWS_FINAL


## 3) Utilities  *(unchanged)*

In [ ]:
def setup_logger(log_dir: str) -> logging.Logger:
    logger = logging.getLogger(f'sharpe_lstm_combined_{os.path.basename(log_dir)}')
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter('[%(asctime)s] [%(levelname)s] %(message)s')
    ch  = logging.StreamHandler()
    ch.setFormatter(fmt)
    logger.addHandler(ch)
    logger.propagate = False
    return logger


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def portfolio_metrics_from_returns(rets: np.ndarray, freq: int = 252) -> Dict[str, float]:
    rets = np.asarray(rets, dtype=np.float64)
    rets = rets[np.isfinite(rets)]
    if rets.size == 0:
        return dict(n_days=0, mean_daily=np.nan, ann_return=np.nan,
                    ann_vol=np.nan, sharpe=np.nan, cum_return=np.nan, max_drawdown=np.nan)
    wealth  = np.cumprod(1.0 + rets)
    mean_d  = float(np.mean(rets))
    vol_d   = float(np.std(rets, ddof=0))
    ann_ret = float((1.0 + mean_d) ** freq - 1.0)
    ann_vol = float(vol_d * freq ** 0.5)
    sharpe  = float(mean_d / vol_d * freq ** 0.5) if vol_d > 1e-12 else np.nan
    peak    = np.maximum.accumulate(wealth)
    max_dd  = float(np.min(wealth / peak - 1.0))
    return dict(n_days=int(rets.size), mean_daily=mean_d, ann_return=ann_ret,
                ann_vol=ann_vol, sharpe=sharpe,
                cum_return=float(wealth[-1] - 1.0), max_drawdown=max_dd)


def _save_cumulative_returns_plot(
    out_dir: str, prefix: str, dates: np.ndarray, port_rets: np.ndarray
):
    os.makedirs(out_dir, exist_ok=True)
    valid = np.isfinite(port_rets)
    if valid.sum() == 0:
        return
    wealth = np.cumprod(1.0 + port_rets[valid])
    plt.figure(figsize=(10, 5))
    plt.plot(dates[valid], wealth)
    plt.title(f'{prefix} | portfolio cumulative return')
    plt.xlabel('date'); plt.ylabel('wealth (1 = initial)')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'{prefix}_cumret.png'))
    plt.close()

## 4) Walk-forward splits  *(unchanged)*

In [ ]:
_CV_PERIODS = [
    ('2015-07-01', '2020-06-30', '2020-07-01', '2021-06-30'),
    ('2015-07-01', '2021-06-30', '2021-07-01', '2022-06-30'),
    ('2015-07-01', '2022-06-30', '2022-07-01', '2023-06-30'),
]


def make_walk_forward_splits(dates_index: pd.DatetimeIndex) -> List[Tuple[np.ndarray, np.ndarray]]:
    splits = []
    for tr_start, tr_end, v_start, v_end in _CV_PERIODS:
        train_idx = np.where(
            (dates_index >= pd.Timestamp(tr_start)) & (dates_index <= pd.Timestamp(tr_end))
        )[0]
        val_idx = np.where(
            (dates_index >= pd.Timestamp(v_start)) & (dates_index <= pd.Timestamp(v_end))
        )[0]
        if len(train_idx) == 0 or len(val_idx) == 0:
            print(f'[SKIP SPLIT] train={tr_start}..{tr_end} val={v_start}..{v_end}')
            continue
        splits.append((train_idx, val_idx))
    if not splits:
        raise ValueError('No valid walk-forward splits.')
    return splits


def split_train_for_early_stop(
    train_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    es_years: int = 1,
    min_train_days: int = 252,
) -> Tuple[np.ndarray, np.ndarray]:
    train_dates = dates_index[train_idx]
    es_start    = train_dates[-1] - pd.DateOffset(years=es_years) + pd.Timedelta(days=1)
    es_mask     = train_dates >= es_start
    core_idx    = train_idx[~es_mask]
    es_idx      = train_idx[es_mask]
    if len(core_idx) < min_train_days or len(es_idx) < min_train_days // 2:
        cut      = min(max(min_train_days, len(train_idx) - 252 * es_years), len(train_idx) - 1)
        core_idx = train_idx[:cut]
        es_idx   = train_idx[cut:]
    if len(core_idx) == 0 or len(es_idx) == 0:
        raise ValueError('Failed to create non-empty early-stopping split.')
    return core_idx, es_idx

## 5) Load frozen configs from news and reddit studies

In [ ]:
def _load_best_sentiment_trial(db_path: str, study_name: str) -> optuna.trial.FrozenTrial:
    """
    Load the best completed trial from a sentiment study DB.
    Raises ValueError if the study has no completed trials.
    """
    storage = f'sqlite:///{db_path}'
    study   = optuna.load_study(study_name=study_name, storage=storage)
    completed = [
        t for t in study.trials
        if t.state == TrialState.COMPLETE
        and t.value is not None
        and np.isfinite(t.value)
    ]
    if not completed:
        raise ValueError(f'No completed trials in study {study_name!r} at {db_path}')
    return max(completed, key=lambda t: t.value)


def load_combined_frozen_config(
    news_db_path: str,
    reddit_db_path: str,
) -> Dict[str, Any]:
    """
    Load the best trial from each sentiment study and build a single combined
    frozen config for use in the combined model.

    Technical base features are the UNION of both frozen pick sets
    (deduplicated, order-preserving: news picks first, then any extras from reddit).
    Market feature: included if either study used it.

    Returns a dict with:
      'picks'              : merged feature picks (Market = VIXIndexClose if either used it)
      'per_asset_feats'    : deduplicated flat feature list
      'news_sentiment_cols': List[str]  suffixes for news columns
      'news_nan_strategy'  : str
      'reddit_sentiment_cols': List[str]  suffixes for reddit columns
      'reddit_nan_strategy': str
      'news_trial_number'  : int
      'news_sharpe'        : float
      'reddit_trial_number': int
      'reddit_sharpe'      : float
    """
    print('Loading best news trial ...')
    news_trial   = _load_best_sentiment_trial(news_db_path,   'sharpe_lstm_sentiment')
    print(f'  Trial #{news_trial.number}  val_sharpe={news_trial.value:.4f}')
    print(f'  frozen_picks:      {news_trial.user_attrs["frozen_picks"]}')
    print(f'  sentiment_cols:    {news_trial.user_attrs["sentiment_picks"]["sentiment_cols"]}')
    print(f'  nan_strategy:      {news_trial.user_attrs["news_nan_strategy"]}')

    print('Loading best reddit trial ...')
    reddit_trial = _load_best_sentiment_trial(reddit_db_path, 'sharpe_lstm_reddit')
    print(f'  Trial #{reddit_trial.number}  val_sharpe={reddit_trial.value:.4f}')
    print(f'  frozen_picks:      {reddit_trial.user_attrs["frozen_picks"]}')
    print(f'  sentiment_cols:    {reddit_trial.user_attrs["sentiment_picks"]["sentiment_cols"]}')
    print(f'  nan_strategy:      {reddit_trial.user_attrs["reddit_nan_strategy"]}')

    news_picks   = news_trial.user_attrs['frozen_picks']
    reddit_picks = reddit_trial.user_attrs['frozen_picks']

    # ── Merge technical picks ─────────────────────────────────────────────
    # Start from news picks, then incorporate any extra features from reddit.
    # For each bucket: if reddit has more indicators (e.g. BBands = 2 cols vs 1),
    # keep both by extending the list.
    def _feats_from_picks(picks):
        feats = ['log_ret_1d']
        for f in picks.get('OHLC', []):
            feats.append(f)
        for bucket in ['Trend', 'Momentum', 'Volume', 'Volatility']:
            val = picks[bucket]
            feats.extend(val if isinstance(val, list) else [val])
        return feats

    news_base   = _feats_from_picks(news_picks)
    reddit_base = _feats_from_picks(reddit_picks)

    # Union: news feats first, append reddit-only extras at the end
    seen = set()
    per_asset_feats = []
    for f in news_base + reddit_base:
        if f not in seen:
            per_asset_feats.append(f)
            seen.add(f)

    # Market: use VIX if either study used it
    use_market = (
        news_picks.get('Market') is not None
        or reddit_picks.get('Market') is not None
    )
    # Build a combined picks dict for required_df_columns compatibility
    merged_picks = {
        'Market': 'VIXIndexClose' if use_market else None,
        # Keep news picks as canonical for the other buckets (used only for
        # required_df_columns column listing — per_asset_feats is the truth)
        **{k: v for k, v in news_picks.items() if k != 'Market'},
    }

    print(f'\nCombined per_asset_feats ({len(per_asset_feats)}): {per_asset_feats}')
    print(f'Market: {merged_picks["Market"]}')

    news_sent   = news_trial.user_attrs['sentiment_picks']['sentiment_cols']
    reddit_sent = reddit_trial.user_attrs['sentiment_picks']['sentiment_cols']
    print(f'News sentiment cols   ({len(news_sent)}):   {news_sent}')
    print(f'Reddit sentiment cols ({len(reddit_sent)}): {reddit_sent}')

    return {
        'picks':                merged_picks,
        'per_asset_feats':      per_asset_feats,
        'news_sentiment_cols':  news_sent,
        'news_nan_strategy':    news_trial.user_attrs['news_nan_strategy'],
        'reddit_sentiment_cols': reddit_sent,
        'reddit_nan_strategy':  reddit_trial.user_attrs['reddit_nan_strategy'],
        'news_trial_number':    news_trial.number,
        'news_sharpe':          news_trial.value,
        'reddit_trial_number':  reddit_trial.number,
        'reddit_sharpe':        reddit_trial.value,
    }


# ── Run at notebook startup ────────────────────────────────────────────────
FROZEN_CONFIG = load_combined_frozen_config(
    news_db_path=NEWS_DB_PATH,
    reddit_db_path=REDDIT_DB_PATH,
)

## 6) NaN strategy helpers  *(unchanged from v3.3)*

In [ ]:
def apply_nan_strategy(
    X_sent: np.ndarray,       # [T, S]  raw sentiment values
    nan_flag: np.ndarray,     # [T]  1 = feed down
    strategy: str,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Clean sentiment values on days where the feed was down.

    Returns:
      X_out      : [T, S]  cleaned values
      loss_mask  : [T] bool  True = exclude from loss (only for 'mask_asset')
    """
    missing   = nan_flag.astype(bool)
    X_out     = X_sent.copy().astype(np.float32)
    loss_mask = np.zeros(len(nan_flag), dtype=bool)

    if strategy == 'zero_fill':
        X_out[missing] = 0.0
    elif strategy == 'carry_forward':
        X_out[missing] = np.nan
        df_tmp = pd.DataFrame(X_out).ffill().fillna(0.0)
        X_out  = df_tmp.to_numpy(dtype=np.float32)
    elif strategy == 'mask_asset':
        X_out[missing] = 0.0
        loss_mask       = missing
    else:
        raise ValueError(f'Unknown nan_strategy: {strategy!r}')

    return X_out, loss_mask

## 7) Feature engineering  *(unchanged)*

In [ ]:
def get_per_asset_feature_list(picks: Dict[str, Any]) -> List[str]:
    feats = ['log_ret_1d']
    for f in picks.get('OHLC', []):
        feats.append(f)
    for bucket in ['Trend', 'Momentum', 'Volume', 'Volatility']:
        val = picks[bucket]
        feats.extend(val if isinstance(val, list) else [val])
    seen, out = set(), []
    for f in feats:
        if f not in seen:
            out.append(f)
            seen.add(f)
    return out


def required_df_columns_combined(
    assets: List[str],
    per_asset_feats: List[str],
    picks: Dict[str, Any],
    news_sentiment_cols: List[str],
    reddit_sentiment_cols: List[str],
) -> List[str]:
    """List all DataFrame columns consumed by build_raw_arrays_combined."""
    cols = []
    for a in assets:
        cols.append(f'{a}_Close')
        for f in per_asset_feats:
            if f != 'log_ret_1d':
                cols.append(f'{a}_{f}')
        for sf in news_sentiment_cols:
            cols.append(f'{a}_{sf}')
        for sf in reddit_sentiment_cols:
            cols.append(f'{a}_{sf}')
    cols.append('_NEWS_NAN')    # single global column
    cols.append('_REDDIT_NAN')  # single global column
    if picks.get('Market') is not None:
        cols.append(picks['Market'])
    return list(dict.fromkeys(cols))  # deduplicated, order-preserving

## 8) Raw arrays & transforms  *(extended for combined sentiment)*

In [ ]:
def build_raw_arrays_combined(
    df: pd.DataFrame,
    assets: List[str],
    frozen_config: Dict[str, Any],
    logger: logging.Logger,
    spreads: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], Optional[np.ndarray]]:
    """
    Build X_raw with the combined feature layout per asset:

      [base_feats (union) | news_sentiment_cols | _NEWS_NAN | reddit_sentiment_cols | _REDDIT_NAN]

    Both _NEWS_NAN and _REDDIT_NAN are SINGLE GLOBAL columns — read once each,
    broadcast identically to all N assets.

    NaN strategies are applied independently per feed using their frozen settings.
    For 'mask_asset': y_mask is narrowed when EITHER feed is down.

    Returns:
      X_raw      : [T, N, F_base + S_news + 1 + S_reddit + 1]
      y_raw      : [T, N]  next-day net returns
      y_mask     : [T, N]  availability mask
      feat_names : List[str]
      market_raw : [T, 1] or None
    """
    per_asset_feats      = frozen_config['per_asset_feats']
    picks                = frozen_config['picks']
    news_sentiment_cols  = frozen_config['news_sentiment_cols']
    news_nan_strategy    = frozen_config['news_nan_strategy']
    reddit_sentiment_cols = frozen_config['reddit_sentiment_cols']
    reddit_nan_strategy  = frozen_config['reddit_nan_strategy']

    cols = required_df_columns_combined(
        assets, per_asset_feats, picks, news_sentiment_cols, reddit_sentiment_cols)
    missing = [c for c in cols if c not in df.columns]
    if missing:
        logger.error(f'Missing columns ({len(missing)}): {missing[:25]}')
        raise KeyError('Dataset missing required columns.')

    if spreads is None:
        spreads = np.zeros(len(assets), dtype=np.float32)
    spreads = np.asarray(spreads, dtype=np.float32)
    if spreads.shape[0] != len(assets):
        raise ValueError(f'spreads length {len(spreads)} != n_assets {len(assets)}')

    T  = len(df)
    N  = len(assets)
    F  = len(per_asset_feats)          # base features
    Sn = len(news_sentiment_cols)      # news sentiment indicators
    Sr = len(reddit_sentiment_cols)    # reddit sentiment indicators
    # Layout: base | news_sent | _NEWS_NAN | reddit_sent | _REDDIT_NAN
    F_total = F + Sn + 1 + Sr + 1

    closes = np.zeros((T, N), dtype=np.float32)
    X_raw  = np.zeros((T, N, F_total), dtype=np.float32)

    # ── Load global NaN flags ONCE ────────────────────────────────────────
    news_nan_flag   = df['_NEWS_NAN'].to_numpy(np.float32)    # [T]
    reddit_nan_flag = df['_REDDIT_NAN'].to_numpy(np.float32)  # [T]

    # Compute global loss masks per feed
    _, news_loss_mask = apply_nan_strategy(
        np.zeros((T, max(Sn, 1)), dtype=np.float32), news_nan_flag, news_nan_strategy)
    _, reddit_loss_mask = apply_nan_strategy(
        np.zeros((T, max(Sr, 1)), dtype=np.float32), reddit_nan_flag, reddit_nan_strategy)
    # Combined: mask day if EITHER feed says to mask
    combined_loss_mask = news_loss_mask | reddit_loss_mask  # [T] bool

    for i, a in enumerate(assets):
        c = df[f'{a}_Close'].to_numpy(np.float32)
        closes[:, i] = c

        # ── Base features ─────────────────────────────────────────────────
        for j, f in enumerate(per_asset_feats):
            if f == 'log_ret_1d':
                lr    = np.full_like(c, np.nan)
                valid = (c[1:] > 0) & (c[:-1] > 0) & np.isfinite(c[1:]) & np.isfinite(c[:-1])
                lr[1:][valid] = np.log(c[1:][valid] / c[:-1][valid])
                X_raw[:, i, j] = lr
            elif f in ('Open', 'High', 'Low'):
                raw   = df[f'{a}_{f}'].to_numpy(np.float32)
                rel   = np.full_like(raw, np.nan)
                valid = (raw > 0) & (c > 0) & np.isfinite(raw) & np.isfinite(c)
                rel[valid] = np.log(raw[valid] / c[valid])
                X_raw[:, i, j] = rel
            elif f.startswith('SMA'):
                sma   = df[f'{a}_{f}'].to_numpy(np.float32)
                rel   = np.full_like(sma, np.nan)
                valid = (sma > 0) & (c > 0) & np.isfinite(sma) & np.isfinite(c)
                rel[valid] = np.log(c[valid] / sma[valid])
                X_raw[:, i, j] = rel
            else:
                X_raw[:, i, j] = df[f'{a}_{f}'].to_numpy(np.float32)

        # ── News sentiment ────────────────────────────────────────────────
        if Sn > 0:
            sent_raw = np.stack(
                [df[f'{a}_{sf}'].to_numpy(np.float32) for sf in news_sentiment_cols], axis=1)
            sent_clean, _ = apply_nan_strategy(sent_raw, news_nan_flag, news_nan_strategy)
            X_raw[:, i, F:F + Sn] = sent_clean
        # _NEWS_NAN flag
        X_raw[:, i, F + Sn] = news_nan_flag

        # ── Reddit sentiment ──────────────────────────────────────────────
        if Sr > 0:
            sent_raw = np.stack(
                [df[f'{a}_{sf}'].to_numpy(np.float32) for sf in reddit_sentiment_cols], axis=1)
            sent_clean, _ = apply_nan_strategy(sent_raw, reddit_nan_flag, reddit_nan_strategy)
            X_raw[:, i, F + Sn + 1:F + Sn + 1 + Sr] = sent_clean
        # _REDDIT_NAN flag
        X_raw[:, i, F + Sn + 1 + Sr] = reddit_nan_flag

    # ── Targets ───────────────────────────────────────────────────────────
    y_raw  = np.full((T, N), np.nan, dtype=np.float32)
    y_mask = np.zeros((T, N), dtype=bool)
    for i in range(N):
        c       = closes[:, i]
        valid   = np.isfinite(c[:-1]) & np.isfinite(c[1:]) & (c[:-1] > 0) & (c[1:] > 0)
        raw_ret = np.full(T - 1, np.nan, dtype=np.float32)
        raw_ret[valid] = (c[1:][valid] / c[:-1][valid]) - 1.0
        net_ret = np.where(
            np.isfinite(raw_ret),
            np.sign(raw_ret) * np.maximum(np.abs(raw_ret) - float(spreads[i]), 0.0),
            np.nan,
        ).astype(np.float32)
        y_raw[:-1, i]  = net_ret
        y_mask[:-1, i] = valid

    # Narrow y_mask by combined loss mask
    y_mask = y_mask & ~combined_loss_mask[:, None]

    market = None
    if picks.get('Market') is not None:
        market = df[picks['Market']].to_numpy(np.float32).reshape(-1, 1)

    # Feature names in layout order
    feat_names = (
        list(per_asset_feats)
        + list(news_sentiment_cols)
        + ['_NEWS_NAN']
        + list(reddit_sentiment_cols)
        + ['_REDDIT_NAN']
    )
    return X_raw, y_raw, y_mask, feat_names, market


def fit_transforms_on_train(
    X_train: np.ndarray,
    feat_names: List[str],
    market_train: Optional[np.ndarray],
) -> Tuple[List[Dict], Optional[Dict]]:
    """
    Fit per-feature z-score normaliser on training window only.
    Both _NEWS_NAN and _REDDIT_NAN are binary flags — identity transform (mu=0, std=1).
    """
    fitted = []
    for f_idx, fname in enumerate(feat_names):
        if fname in ('_NEWS_NAN', '_REDDIT_NAN'):
            fitted.append({'mu': 0.0, 'std': 1.0})  # identity
            continue
        vals = X_train[:, :, f_idx].ravel()
        vals = vals[np.isfinite(vals)]
        mu   = float(vals.mean()) if len(vals) > 0 else 0.0
        std  = float(vals.std())  if len(vals) > 0 else 1.0
        fitted.append({'mu': mu, 'std': max(std, 1e-8)})

    market_ft = None
    if market_train is not None:
        vals = market_train.ravel()
        vals = vals[np.isfinite(vals)]
        mu   = float(vals.mean()) if len(vals) > 0 else 0.0
        std  = float(vals.std())  if len(vals) > 0 else 1.0
        market_ft = {'mu': mu, 'std': max(std, 1e-8)}
    return fitted, market_ft


def apply_transforms(
    X: np.ndarray,
    fitted: List[Dict],
    market: Optional[np.ndarray],
    market_ft: Optional[Dict],
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    X_out = X.copy()
    for f_idx, ft in enumerate(fitted):
        X_out[:, :, f_idx] = (X[:, :, f_idx] - ft['mu']) / ft['std']
    X_out = np.nan_to_num(X_out, nan=0.0, posinf=0.0, neginf=0.0)
    m_out = None
    if market is not None and market_ft is not None:
        m_out = (market - market_ft['mu']) / market_ft['std']
        m_out = np.nan_to_num(m_out, nan=0.0, posinf=0.0, neginf=0.0)
    return X_out, m_out


def compute_sample_validity(
    X_raw: np.ndarray, y_mask: np.ndarray, lookback: int, min_valid_assets: int = 1
) -> np.ndarray:
    finite_feats = np.isfinite(X_raw).all(axis=2)
    sample_ok    = np.zeros(len(X_raw), dtype=bool)
    for t in range(lookback - 1, len(X_raw) - 1):
        win_ok    = finite_feats[t - lookback + 1: t + 1].all()
        target_ok = int(y_mask[t].sum()) >= min_valid_assets
        sample_ok[t] = bool(win_ok and target_ok)
    return sample_ok

## 9) Dataset & DataLoader  *(unchanged)*

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, X, y, y_mask, valid_t, lookback, market=None):
        self.X        = X
        self.y        = y
        self.y_mask   = y_mask
        self.valid_t  = np.asarray(valid_t, dtype=np.int64)
        self.lookback = int(lookback)
        self.market   = market

    def __len__(self): return len(self.valid_t)

    def __getitem__(self, idx):
        t     = int(self.valid_t[idx])
        x     = self.X[t - self.lookback + 1: t + 1].copy()
        if self.market is not None:
            m = self.market[t - self.lookback + 1: t + 1]
            m = np.repeat(m[:, None, :], x.shape[1], axis=1)
            x = np.concatenate([x, m], axis=2)
        y     = self.y[t].copy()
        avail = self.y_mask[t].copy().astype(np.float32)
        x     = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        y     = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
        return (
            torch.tensor(x,     dtype=torch.float32),
            torch.tensor(y,     dtype=torch.float32),
            torch.tensor(avail, dtype=torch.float32),
            torch.tensor(t,     dtype=torch.long),
        )


def make_loader(X, y, y_mask, valid_t, lookback, batch_size, market, shuffle):
    ds = SequenceDataset(X=X, y=y, y_mask=y_mask, valid_t=valid_t,
                         lookback=lookback, market=market)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

## 10) Model — WeightLSTM  *(unchanged)*

In [ ]:
class CrossAssetMixer(nn.Module):
    def __init__(self, d_model, num_heads, dropout, num_layers):
        super().__init__()
        if num_layers <= 0:
            self.net = nn.Identity()
        else:
            enc_layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=num_heads,
                dim_feedforward=max(4 * d_model, 64),
                dropout=dropout, batch_first=True,
                activation='gelu', norm_first=True,
            )
            self.net = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

    def forward(self, x): return self.net(x)


class WeightLSTM(nn.Module):
    """
    Shared-encoder LSTM that outputs portfolio weights directly.
    weight_mode: 'softmax' (long-only) or 'unconstrained' (allows shorts).
    Input dimension adapts automatically to the combined feature set.
    """
    def __init__(self, input_dim_per_asset, num_assets, hidden_size, num_layers,
                 dropout, bidirectional=False, encoder_proj=0,
                 asset_mixer_layers=1, asset_mixer_heads=4,
                 fusion_hidden=64, weight_mode='softmax'):
        super().__init__()
        self.num_assets  = num_assets
        self.weight_mode = weight_mode
        lstm_dropout     = dropout if num_layers > 1 else 0.0
        self.encoder     = nn.LSTM(
            input_size=input_dim_per_asset, hidden_size=hidden_size,
            num_layers=num_layers, dropout=lstm_dropout,
            batch_first=True, bidirectional=bidirectional,
        )
        enc_dim = hidden_size * (2 if bidirectional else 1)
        if encoder_proj > 0:
            self.asset_proj = nn.Sequential(
                nn.Linear(enc_dim, encoder_proj), nn.GELU(), nn.Dropout(dropout))
            asset_dim = encoder_proj
        else:
            self.asset_proj = nn.Identity()
            asset_dim       = enc_dim
        if asset_mixer_layers > 0:
            valid_heads = [h for h in [1, 2, 4, 8] if asset_dim % h == 0]
            if not valid_heads:          asset_mixer_heads = 1
            elif asset_mixer_heads not in valid_heads: asset_mixer_heads = max(valid_heads)
        self.asset_mixer = CrossAssetMixer(
            d_model=asset_dim,
            num_heads=asset_mixer_heads if asset_mixer_layers > 0 else 1,
            dropout=dropout, num_layers=asset_mixer_layers,
        )
        fusion_in = num_assets * asset_dim
        self.head = (
            nn.Sequential(
                nn.Linear(fusion_in, fusion_hidden), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(fusion_hidden, num_assets))
            if fusion_hidden > 0
            else nn.Linear(fusion_in, num_assets)
        )

    def forward(self, x: torch.Tensor, avail: torch.Tensor) -> torch.Tensor:
        B, L, N, F = x.shape
        x      = x.permute(0, 2, 1, 3).contiguous().reshape(B * N, L, F)
        out, _ = self.encoder(x)
        h      = self.asset_proj(out[:, -1, :])
        h      = h.reshape(B, N, -1)
        z      = self.asset_mixer(h)
        logits = self.head(z.reshape(B, -1))
        return self._to_weights(logits, avail)

    def _to_weights(self, logits, avail):
        mask_val = torch.finfo(logits.dtype).min / 2
        masked   = logits + (1.0 - avail) * mask_val
        if self.weight_mode == 'softmax':
            w = torch.softmax(masked, dim=-1) * avail
        elif self.weight_mode == 'unconstrained':
            w = masked * avail
            w = w / w.abs().sum(dim=-1, keepdim=True).clamp(min=1e-8)
        else:
            raise ValueError(f'Unknown weight_mode: {self.weight_mode}')
        return w

## 11) Loss & training loop  *(unchanged)*

In [ ]:
def sharpe_loss(weights, returns, avail, eps=1e-8, annualize=False, freq=252):
    port_ret = (weights * returns * avail).sum(dim=-1)
    mean_ret = port_ret.mean()
    std_ret  = port_ret.std(unbiased=False).clamp(min=eps)
    sharpe   = mean_ret / std_ret
    if annualize:
        sharpe = sharpe * (freq ** 0.5)
    if not torch.isfinite(sharpe):
        raise ValueError('Sharpe became non-finite.')
    return -sharpe


@torch.no_grad()
def evaluate_model(model: WeightLSTM, loader: DataLoader, device: torch.device) -> Dict[str, Any]:
    model.eval()
    port_rets, weights_all, t_idx_all = [], [], []
    for xb, yb, avail_b, tb in loader:
        xb, yb, avail_b = xb.to(device), yb.to(device), avail_b.to(device)
        w  = model(xb, avail_b)
        pr = (w * yb * avail_b).sum(dim=-1)
        port_rets.append(pr.cpu().numpy())
        weights_all.append(w.cpu().numpy())
        t_idx_all.append(tb.numpy())
    port_rets   = np.concatenate(port_rets)   if port_rets   else np.array([])
    weights_all = np.concatenate(weights_all) if weights_all else np.empty((0, 0))
    t_idx_all   = np.concatenate(t_idx_all)   if t_idx_all   else np.array([], dtype=np.int64)
    return {'metrics': portfolio_metrics_from_returns(port_rets),
            'port_rets': port_rets, 'weights': weights_all, 't_idx': t_idx_all}


def train_one_model(
    X, y, y_mask, market, train_t, es_t, lookback,
    input_dim_per_asset, num_assets, params, seed, logger,
) -> Tuple[WeightLSTM, Dict]:
    seed_everything(seed)
    device       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_loader = make_loader(X, y, y_mask, train_t, lookback,
                               params['batch_size'], market, shuffle=True)
    es_loader    = make_loader(X, y, y_mask, es_t, lookback,
                               params['batch_size'], market, shuffle=False)

    model = WeightLSTM(
        input_dim_per_asset=input_dim_per_asset, num_assets=num_assets,
        hidden_size=params['hidden_size'],    num_layers=params['num_layers'],
        dropout=params['dropout'],            bidirectional=params['bidirectional'],
        encoder_proj=params['encoder_proj'],
        asset_mixer_layers=params['asset_mixer_layers'],
        asset_mixer_heads=params['asset_mixer_heads'],
        fusion_hidden=params['fusion_hidden'], weight_mode=params['weight_mode'],
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])
    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        if params['scheduler'] == 'plateau' else None
    )

    best_state, best_score, best_epoch, patience_count = None, np.inf, -1, 0

    for epoch in range(1, params['max_epochs'] + 1):
        model.train()
        batch_losses = []
        for xb, yb, avail_b, _ in train_loader:
            xb, yb, avail_b = xb.to(device), yb.to(device), avail_b.to(device)
            if not torch.isfinite(xb).all(): raise ValueError('Non-finite input.')
            if avail_b.sum() <= 0: continue
            optimizer.zero_grad()
            w    = model(xb, avail_b)
            if not torch.isfinite(w).all(): raise ValueError('Non-finite weights.')
            loss = sharpe_loss(w, yb, avail_b, eps=params['sharpe_eps'])
            if not torch.isfinite(loss): raise ValueError('Non-finite loss.')
            loss.backward()
            if params['grad_clip'] > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), params['grad_clip'])
            optimizer.step()
            batch_losses.append(float(loss.item()))

        es_eval       = evaluate_model(model, es_loader, device)
        es_sharpe     = es_eval['metrics']['sharpe']
        es_neg_sharpe = -es_sharpe if np.isfinite(es_sharpe) else np.inf
        train_loss    = float(np.mean(batch_losses)) if batch_losses else np.nan
        logger.info(f'[TRAIN] seed={seed} epoch={epoch:03d} '
                    f'train_loss={train_loss:.6f} es_sharpe={es_sharpe:.4f}')

        if scheduler is not None: scheduler.step(es_neg_sharpe)

        if np.isfinite(es_neg_sharpe) and es_neg_sharpe < (best_score - params['min_delta']):
            best_score, best_epoch, best_state, patience_count = (
                es_neg_sharpe, epoch, copy.deepcopy(model.state_dict()), 0)
        else:
            patience_count += 1

        if epoch >= params['min_epochs'] and patience_count >= params['patience']:
            logger.info(f'[EARLY STOP] seed={seed} best_epoch={best_epoch} '
                        f'best_sharpe={-best_score:.4f}')
            break

    if best_state is None:
        raise ValueError('No valid best_state found.')
    model.load_state_dict(best_state)
    return model, {}


def train_and_eval_one_split_seed(
    X_transformed, y, y_mask, market_transformed,
    asset_names, dates_index, train_idx, val_idx,
    lookback, seed, model_params, trial_dir, logger, split_i,
    save_artifacts=False,
) -> Dict[str, Any]:
    train_core_idx, es_idx = split_train_for_early_stop(train_idx, dates_index)
    sample_valid = compute_sample_validity(
        X_transformed, y_mask, lookback, min_valid_assets=max(1, len(asset_names) // 3))
    train_t = train_core_idx[sample_valid[train_core_idx]]
    es_t    = es_idx[sample_valid[es_idx]]
    val_t   = val_idx[sample_valid[val_idx]]

    if len(train_t) < 64 or len(es_t) < 32 or len(val_t) < 32:
        raise optuna.exceptions.TrialPruned()

    num_assets = len(asset_names)
    input_dim  = X_transformed.shape[2] + (1 if market_transformed is not None else 0)
    logger.info(f'[SPLIT {split_i}] seed={seed} train={len(train_t)} '
                f'es={len(es_t)} val={len(val_t)} input_dim={input_dim}')

    model, _ = train_one_model(
        X=X_transformed, y=y, y_mask=y_mask, market=market_transformed,
        train_t=train_t, es_t=es_t, lookback=lookback,
        input_dim_per_asset=input_dim, num_assets=num_assets,
        params=model_params, seed=seed, logger=logger,
    )
    device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    val_loader = make_loader(X_transformed, y, y_mask, val_t, lookback,
                             model_params['batch_size'], market_transformed, shuffle=False)
    val_eval   = evaluate_model(model, val_loader, device)
    vm         = val_eval['metrics']
    logger.info(f"[VAL] split={split_i} seed={seed} sharpe={vm['sharpe']:.4f} "
                f"ann_ret={vm['ann_return']:.4f}")
    return {'metrics': vm, 'port_rets': val_eval['port_rets'],
            'weights': val_eval['weights'], 't_idx': val_eval['t_idx']}

## 12) Optuna objective — hyperparameters only

In [ ]:
# All feature choices are frozen from FROZEN_CONFIG.
# Optuna searches ONLY architecture + training hyperparameters.

def objective_factory(
    df: pd.DataFrame,
    assets: List[str],
    frozen_config: Dict[str, Any],
    base_log_dir: str,
    seeds: List[int],
    spreads: Optional[np.ndarray] = None,
):
    dates_index = pd.DatetimeIndex(df.index)
    splits      = make_walk_forward_splits(dates_index)

    # Pre-build arrays once — same for every trial (features are frozen)
    _logger = setup_logger(base_log_dir)
    X_raw, y_raw, y_mask, feat_names, market_raw = build_raw_arrays_combined(
        df=df, assets=assets, frozen_config=frozen_config,
        logger=_logger, spreads=spreads,
    )
    _logger.info(f'Combined feature dim per asset: {X_raw.shape[2]}  '
                 f'(feat_names={feat_names})')

    def objective(trial: optuna.Trial) -> float:
        trial_dir = os.path.join(base_log_dir, f'trial_{trial.number:05d}')
        logger    = setup_logger(trial_dir)
        logger.info('=' * 80)
        logger.info(f'TRIAL {trial.number} — hyperparameter search only')
        logger.info(f'Feature dim per asset: {X_raw.shape[2]}')

        # ── Architecture hyperparameters ──────────────────────────────────
        lookback           = trial.suggest_categorical('lookback',           [5, 10, 22])
        hidden_size        = trial.suggest_categorical('hidden_size',        [32, 64, 128])
        num_layers         = trial.suggest_int('num_layers',                 1, 2)
        dropout            = trial.suggest_float('dropout',                  0.0, 0.3)
        bidirectional      = bool(trial.suggest_categorical('bidirectional', [0, 1]))
        encoder_proj       = trial.suggest_categorical('encoder_proj',       [0, 32, 64])
        asset_mixer_layers = trial.suggest_categorical('asset_mixer_layers', [0, 1, 2])
        asset_mixer_heads  = trial.suggest_categorical('asset_mixer_heads',  [1, 2, 4])
        fusion_hidden      = trial.suggest_categorical('fusion_hidden',      [0, 64, 128])
        weight_mode        = trial.suggest_categorical('weight_mode',        ['softmax', 'unconstrained'])

        # ── Training hyperparameters ──────────────────────────────────────
        learning_rate = trial.suggest_float('learning_rate', 1e-4, 2e-3, log=True)
        weight_decay  = trial.suggest_float('weight_decay',  1e-6, 1e-3, log=True)
        batch_size    = trial.suggest_categorical('batch_size',  [32, 64, 128])
        max_epochs    = trial.suggest_categorical('max_epochs',  [20, 30, 40])
        patience      = trial.suggest_categorical('patience',    [5, 7])
        min_epochs    = trial.suggest_categorical('min_epochs',  [5, 8])
        grad_clip     = trial.suggest_categorical('grad_clip',   [0.5, 1.0, 2.0])
        scheduler     = trial.suggest_categorical('scheduler',   ['none', 'plateau'])
        sharpe_eps    = trial.suggest_categorical('sharpe_eps',  [1e-6, 1e-8])
        min_delta     = trial.suggest_categorical('min_delta',   [0.0, 1e-4])

        model_params = dict(
            hidden_size=hidden_size,         num_layers=num_layers,
            dropout=dropout,                 bidirectional=bidirectional,
            encoder_proj=encoder_proj,       asset_mixer_layers=asset_mixer_layers,
            asset_mixer_heads=asset_mixer_heads, fusion_hidden=fusion_hidden,
            weight_mode=weight_mode,         learning_rate=learning_rate,
            weight_decay=weight_decay,       batch_size=batch_size,
            max_epochs=max_epochs,           patience=patience,
            min_epochs=min_epochs,           grad_clip=grad_clip,
            scheduler=scheduler,             sharpe_eps=sharpe_eps,
            min_delta=min_delta,
        )
        logger.info(f'Model params: {model_params}')

        all_sharpes = []
        for split_i, (train_idx, val_idx) in enumerate(splits):
            logger.info('-' * 80)
            logger.info(f'[SPLIT {split_i}] '
                        f'train {df.index[train_idx[0]]}..{df.index[train_idx[-1]]} | '
                        f'val   {df.index[val_idx[0]]}..{df.index[val_idx[-1]]}')

            X_tr_raw       = X_raw[train_idx]
            mkt_tr_raw     = market_raw[train_idx] if market_raw is not None else None
            fitted, mkt_ft = fit_transforms_on_train(X_tr_raw, feat_names, mkt_tr_raw)
            X_trans, mkt_trans = apply_transforms(X_raw, fitted, market_raw, mkt_ft)

            split_scores = []
            for seed in seeds:
                try:
                    res = train_and_eval_one_split_seed(
                        X_transformed=X_trans, y=y_raw, y_mask=y_mask,
                        market_transformed=mkt_trans,
                        asset_names=assets, dates_index=dates_index,
                        train_idx=train_idx, val_idx=val_idx,
                        lookback=lookback, seed=seed, model_params=model_params,
                        trial_dir=trial_dir, logger=logger, split_i=split_i,
                        save_artifacts=False,
                    )
                except ValueError as e:
                    logger.info(f'[PRUNE] Non-finite: {e}')
                    raise optuna.TrialPruned()

                sharpe = res['metrics']['sharpe']
                score  = float(sharpe) if np.isfinite(sharpe) else -999.0
                all_sharpes.append(score)
                split_scores.append(score)
                logger.info(f'[SPLIT {split_i}][SEED {seed}] sharpe={score:.4f} '
                            f"ann_ret={res['metrics']['ann_return']:.4f}")

            split_mean = float(np.mean(split_scores))
            trial.report(split_mean, step=split_i)
            if trial.should_prune():
                logger.info('[PRUNE] MedianPruner.')
                raise optuna.TrialPruned()

        final_score = float(np.mean(all_sharpes))
        trial.set_user_attr('all_sharpes', [float(x) for x in all_sharpes])
        trial.set_user_attr('mean_sharpe', final_score)
        logger.info(f'TRIAL {trial.number} DONE | mean_sharpe={final_score:.4f}')
        return final_score

    return objective


def run_hp_search(
    df: pd.DataFrame,
    assets: List[str],
    frozen_config: Dict[str, Any],
    out_dir: str,
    n_trials: int,
    seeds: Optional[List[int]] = None,
    spreads: Optional[np.ndarray] = None,
) -> optuna.Study:
    os.makedirs(out_dir, exist_ok=True)
    seeds = seeds or [11, 22, 33]

    db_path = os.path.join(out_dir, 'sharpe_lstm_combined.db')
    study   = optuna.create_study(
        study_name='sharpe_lstm_combined',
        direction='maximize',
        storage=f'sqlite:///{db_path}',
        load_if_exists=True,
        sampler=TPESampler(seed=None, multivariate=True, group=True, constant_liar=True),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1),
    )

    objective = objective_factory(
        df=df, assets=assets, frozen_config=frozen_config,
        base_log_dir=out_dir, seeds=seeds, spreads=spreads,
    )
    study.optimize(objective, n_trials=n_trials, gc_after_trial=True, show_progress_bar=True)

    best = study.best_trial
    with open(os.path.join(out_dir, 'best_trial.json'), 'w') as f:
        json.dump({
            'best_value':       best.value,
            'best_params':      best.params,
            'frozen_config':    {
                'per_asset_feats':       frozen_config['per_asset_feats'],
                'news_sentiment_cols':   frozen_config['news_sentiment_cols'],
                'news_nan_strategy':     frozen_config['news_nan_strategy'],
                'reddit_sentiment_cols': frozen_config['reddit_sentiment_cols'],
                'reddit_nan_strategy':   frozen_config['reddit_nan_strategy'],
                'news_sharpe':           frozen_config['news_sharpe'],
                'reddit_sharpe':         frozen_config['reddit_sharpe'],
            },
            'n_trials': len(study.trials),
        }, f, indent=2)

    print('\n=== BEST TRIAL ===')
    print('Mean val Sharpe:', best.value)
    print('Params:',          best.params)
    print('DB:', db_path)
    return study

## 13) Search space inspector  *(unchanged)*

In [ ]:
def estimate_search_space(study: optuna.Study) -> int:
    if not study.trials:
        raise ValueError('Run at least 1 trial first so distributions are populated.')
    distributions         = study.trials[0].distributions
    discrete_combinations = 1
    continuous_params     = []
    print('Search space:')
    for name, dist in distributions.items():
        if isinstance(dist, optuna.distributions.CategoricalDistribution):
            n = len(dist.choices)
            discrete_combinations *= n
            print(f'  {name:30s}  categorical  {n:>4d} choices')
        elif isinstance(dist, optuna.distributions.IntDistribution):
            n = (dist.high - dist.low) // dist.step + 1
            discrete_combinations *= n
            print(f'  {name:30s}  int          {n:>4d} levels  [{dist.low}..{dist.high}]')
        elif isinstance(dist, optuna.distributions.FloatDistribution):
            continuous_params.append(name)
            scale = 'log' if dist.log else 'linear'
            print(f'  {name:30s}  float ({scale:6s})  [{dist.low:.2e}..{dist.high:.2e}]  ← continuous')
    suggested = math.ceil(math.sqrt(discrete_combinations))
    print(f'\nDiscrete combinations : {discrete_combinations}')
    print(f'Continuous params     : {continuous_params}')
    print(f'Suggested N_TRIALS    : ceil(sqrt({discrete_combinations})) = {suggested}')
    return suggested

## 14) Final test evaluation  *(adapted from v3.3)*

In [ ]:
def make_anchor_idx_by_target_date_range(
    dates_index: pd.DatetimeIndex,
    target_start: Optional[str] = None,
    target_end: Optional[str] = None,
) -> np.ndarray:
    anchor_idx   = np.arange(len(dates_index) - 1, dtype=np.int64)
    target_dates = dates_index[1:]
    mask = np.ones(len(anchor_idx), dtype=bool)
    if target_start: mask &= target_dates >= pd.Timestamp(target_start)
    if target_end:   mask &= target_dates <= pd.Timestamp(target_end)
    out = anchor_idx[mask]
    if len(out) == 0:
        raise ValueError(f'No anchors for target range {target_start}..{target_end}')
    return out


def refit_and_predict_block(
    df: pd.DataFrame,
    assets: List[str],
    best_trial: optuna.trial.FrozenTrial,
    frozen_config: Dict[str, Any],
    train_idx: np.ndarray,
    test_idx: np.ndarray,
    dates_index: pd.DatetimeIndex,
    seed: int,
    logger: logging.Logger,
    spreads: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    N      = len(assets)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    lookback     = int(best_trial.params['lookback'])
    model_params = {k: best_trial.params[k] for k in [
        'hidden_size', 'num_layers', 'dropout', 'bidirectional', 'encoder_proj',
        'asset_mixer_layers', 'asset_mixer_heads', 'fusion_hidden', 'weight_mode',
        'learning_rate', 'weight_decay', 'batch_size', 'max_epochs', 'patience',
        'min_epochs', 'grad_clip', 'scheduler',
    ]}
    model_params['bidirectional'] = bool(model_params['bidirectional'])
    model_params['sharpe_eps']    = best_trial.params.get('sharpe_eps', 1e-8)
    model_params['min_delta']     = best_trial.params.get('min_delta',  0.0)

    X_raw, y_raw, y_mask, feat_names, market_raw = build_raw_arrays_combined(
        df=df, assets=assets, frozen_config=frozen_config,
        logger=logger, spreads=spreads,
    )
    fitted, mkt_ft = fit_transforms_on_train(
        X_raw[train_idx], feat_names,
        market_raw[train_idx] if market_raw is not None else None,
    )
    X_trans, mkt_trans = apply_transforms(X_raw, fitted, market_raw, mkt_ft)

    sample_valid = compute_sample_validity(X_trans, y_mask, lookback,
                                           min_valid_assets=max(1, N // 3))
    core_idx, es_idx = split_train_for_early_stop(train_idx, dates_index)
    train_core_t     = core_idx[sample_valid[core_idx]]
    es_t             = es_idx[sample_valid[es_idx]]
    test_t           = test_idx[sample_valid[test_idx]]
    all_train_t      = train_idx[sample_valid[train_idx]]

    if len(train_core_t) < 64 or len(es_t) < 32 or len(test_t) < 32:
        raise ValueError(f'Not enough samples: train_core={len(train_core_t)} '
                         f'es={len(es_t)} test={len(test_t)}')

    input_dim = X_trans.shape[2] + (1 if mkt_trans is not None else 0)
    model, _  = train_one_model(
        X=X_trans, y=y_raw, y_mask=y_mask, market=mkt_trans,
        train_t=train_core_t, es_t=es_t, lookback=lookback,
        input_dim_per_asset=input_dim, num_assets=N,
        params=model_params, seed=seed, logger=logger,
    )

    def _run_eval(t_indices):
        loader = make_loader(X_trans, y_raw, y_mask, t_indices, lookback,
                             model_params['batch_size'], mkt_trans, shuffle=False)
        return evaluate_model(model, loader, device)

    def _to_df(eval_res, t_indices):
        n_days   = len(t_indices)
        w_full   = np.full((n_days, N), np.nan, dtype=np.float32)
        pr_full  = np.full(n_days,      np.nan, dtype=np.float32)
        t_to_pos = {int(t): pos for pos, t in enumerate(t_indices)}
        for i, t in enumerate(eval_res['t_idx']):
            pos = t_to_pos.get(int(t))
            if pos is not None:
                w_full[pos]  = eval_res['weights'][i]
                pr_full[pos] = eval_res['port_rets'][i]
        target_dates = dates_index[t_indices + 1]
        return (pd.DataFrame(w_full,  index=target_dates, columns=assets).sort_index(),
                pd.DataFrame({'portfolio_return': pr_full}, index=target_dates).sort_index())

    train_w_df, train_r_df = _to_df(_run_eval(all_train_t), all_train_t)
    test_w_df,  test_r_df  = _to_df(_run_eval(test_t),      test_t)

    test_pm  = portfolio_metrics_from_returns(test_r_df['portfolio_return'].dropna().to_numpy())
    train_pm = portfolio_metrics_from_returns(train_r_df['portfolio_return'].dropna().to_numpy())
    logger.info(f"[BLOCK] train_sharpe={train_pm['sharpe']:.4f} | "
                f"test_sharpe={test_pm['sharpe']:.4f} "
                f"ann_ret={test_pm['ann_return']:.4f} max_dd={test_pm['max_drawdown']:.4f}")

    return {'test_weights_df': test_w_df, 'test_returns_df': test_r_df, 'test_metrics': test_pm,
            'train_weights_df': train_w_df, 'train_returns_df': train_r_df, 'train_metrics': train_pm}


def final_test_eval_with_midtest_refit(
    df: pd.DataFrame,
    assets: List[str],
    study: optuna.Study,
    frozen_config: Dict[str, Any],
    out_dir: str,
    seed: int = SEED_PORTFOLIO_LSTM_SHARPE_V3_2_X_REDDIT_NEWS,
    train_target_end: str = '2023-06-30',
    test_year1_start: str = '2023-07-01',
    test_year1_end:   str = '2024-06-30',
    test_year2_start: str = '2024-07-01',
    test_year2_end:   str = '2025-06-30',
    spreads: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    final_dir   = os.path.join(out_dir, 'final_test')
    os.makedirs(final_dir, exist_ok=True)
    logger      = setup_logger(final_dir)
    dates_index = pd.DatetimeIndex(df.index)
    best_trial  = study.best_trial

    b1_train = make_anchor_idx_by_target_date_range(dates_index, target_end=train_target_end)
    b1_test  = make_anchor_idx_by_target_date_range(dates_index,
                   target_start=test_year1_start, target_end=test_year1_end)
    b2_train = make_anchor_idx_by_target_date_range(dates_index, target_end=test_year1_end)
    b2_test  = make_anchor_idx_by_target_date_range(dates_index,
                   target_start=test_year2_start, target_end=test_year2_end)

    logger.info(f'Block1 train..{train_target_end} | test {test_year1_start}..{test_year1_end}')
    logger.info(f'Block2 train..{test_year1_end}   | test {test_year2_start}..{test_year2_end}')

    block1 = refit_and_predict_block(
        df=df, assets=assets, best_trial=best_trial, frozen_config=frozen_config,
        train_idx=b1_train, test_idx=b1_test,
        dates_index=dates_index, seed=seed, logger=logger, spreads=spreads)
    block2 = refit_and_predict_block(
        df=df, assets=assets, best_trial=best_trial, frozen_config=frozen_config,
        train_idx=b2_train, test_idx=b2_test,
        dates_index=dates_index, seed=seed, logger=logger, spreads=spreads)

    test_weights_df  = pd.concat([block1['test_weights_df'],  block2['test_weights_df']]).sort_index()
    test_returns_df  = pd.concat([block1['test_returns_df'],  block2['test_returns_df']]).sort_index()
    train_weights_df = block1['train_weights_df']
    train_returns_df = block1['train_returns_df']

    combined_pm = portfolio_metrics_from_returns(
        test_returns_df['portfolio_return'].dropna().to_numpy())

    for name, obj in [('test_weights', test_weights_df), ('test_returns', test_returns_df),
                      ('train_weights', train_weights_df), ('train_returns', train_returns_df)]:
        obj.to_csv(os.path.join(final_dir, f'{name}.csv'))
        logger.info(f'Saved {name}.csv ({len(obj)} rows)')

    with open(os.path.join(final_dir, 'test_metrics.json'), 'w') as f:
        json.dump({
            'block1':    block1['test_metrics'],
            'block2':    block2['test_metrics'],
            'combined':  combined_pm,
            'frozen_config': {
                'per_asset_feats':       frozen_config['per_asset_feats'],
                'news_sentiment_cols':   frozen_config['news_sentiment_cols'],
                'news_nan_strategy':     frozen_config['news_nan_strategy'],
                'reddit_sentiment_cols': frozen_config['reddit_sentiment_cols'],
                'reddit_nan_strategy':   frozen_config['reddit_nan_strategy'],
            },
        }, f, indent=2)

    print('\n=== FINAL TEST RESULTS ===')
    print(f"Block1 (2023-H2): {block1['test_metrics']}")
    print(f"Block2 (2024-H2): {block2['test_metrics']}")
    print(f"Combined:         {combined_pm}")

    return {
        'test_weights_df':       test_weights_df,
        'test_returns_df':       test_returns_df,
        'train_weights_df':      train_weights_df,
        'train_returns_df':      train_returns_df,
        'block1_test_metrics':   block1['test_metrics'],
        'block2_test_metrics':   block2['test_metrics'],
        'combined_test_metrics': combined_pm,
    }

## 15) Main execution

In [ ]:
# ─── Load data ────────────────────────────────────────────────────────────────
df = pd.read_parquet(str(DATASET_PATH))
df.index = pd.to_datetime(df.index)
df = df.sort_index()

with open(str(SPREADS_PATH), 'r') as f:
    spread_dict = json.load(f)
spreads = np.array([spread_dict[a] for a in ASSETS], dtype=np.float32)

print(f'Dataset: {df.shape}  ({df.index[0].date()} → {df.index[-1].date()})')
print(f'Assets:  {ASSETS}')

In [ ]:
# ─── Sanity-check that all frozen columns are present in df ───────────────────
required = required_df_columns_combined(
    assets=ASSETS,
    per_asset_feats=FROZEN_CONFIG['per_asset_feats'],
    picks=FROZEN_CONFIG['picks'],
    news_sentiment_cols=FROZEN_CONFIG['news_sentiment_cols'],
    reddit_sentiment_cols=FROZEN_CONFIG['reddit_sentiment_cols'],
)
missing = [c for c in required if c not in df.columns]
if missing:
    print(f'WARNING: {len(missing)} required columns not found in df.')
    print('First 10:', missing[:10])
else:
    print(f'All {len(required)} required columns present.')

print(f'\nFeature summary:')
print(f'  Base feats:            {FROZEN_CONFIG["per_asset_feats"]}')
print(f'  News sentiment cols:   {FROZEN_CONFIG["news_sentiment_cols"]}')
print(f'  News NaN strategy:     {FROZEN_CONFIG["news_nan_strategy"]}')
print(f'  Reddit sentiment cols: {FROZEN_CONFIG["reddit_sentiment_cols"]}')
print(f'  Reddit NaN strategy:   {FROZEN_CONFIG["reddit_nan_strategy"]}')
print(f'  Market:                {FROZEN_CONFIG["picks"].get("Market")}')

In [ ]:
# ─── Step 1: hyperparameter search ───────────────────────────────────────────
# Set N_TRIALS > 0 before running.
# Features are fully frozen — only architecture + training HPs are searched.
# Tip: run N_TRIALS=1 first, then call estimate_search_space(study).

os.makedirs(OUT_DIR, exist_ok=True)

study = run_hp_search(
    df=df,
    assets=ASSETS,
    frozen_config=FROZEN_CONFIG,
    out_dir=OUT_DIR,
    n_trials=N_TRIALS,
    seeds=MODEL_SEEDS,
    spreads=spreads,
)

suggested = estimate_search_space(study)
print(f'\nYou used N_TRIALS={len(study.trials)}. Suggested ceiling: {suggested}.')

In [ ]:
# ─── Step 2: final test with mid-test refit ───────────────────────────────────
final_result = final_test_eval_with_midtest_refit(
    df=df,
    assets=ASSETS,
    study=study,
    frozen_config=FROZEN_CONFIG,
    out_dir=OUT_DIR,
    seed=FINAL_SEED,
    spreads=spreads,
)

In [ ]:
non_tradable_days = str(PROJECT_ROOT / "01_data/aux/non_tradable_days.txt")
must_have_weights = pd.DataFrame(index=pd.date_range(start="2023-07-01", end="2025-06-30", freq="D"))
non_tradable_dates = pd.to_datetime(
    pd.read_csv(non_tradable_days, header=None)[0]
)
must_have_weights = must_have_weights.drop(non_tradable_dates, errors="ignore")

weights = must_have_weights.join(final_result['test_weights_df'], how="left")

weights = weights.fillna(method="ffill").fillna(method="bfill")[ASSETS].values

In [ ]:
import sys
from pathlib import Path
from src.metrics import PortfolioMetrics

pm = PortfolioMetrics()
print(pm.summary(weights))
pm.plot_weights(weights)
pm.plot_cumulative_returns(weights)


In [ ]:
np.save("LSTMvia-sharpe_reddit+news_weights.npy", weights)